## 🎯 Learning Objectives
* Implement a Proximal Policy Optimization (PPO) agent using a modern reinforcement learning library.
* Configure and interact with a continuous control environment from Gymnasium.
* Train a PPO agent on a complex control task and monitor its learning progress.
* Evaluate the performance of a trained PPO agent and understand key evaluation metrics.
* Gain practical experience with hyperparameter tuning and best practices for PPO.


## Exercise: Train a PPO Agent on a Continuous Control Task

**Lesson ID**: FT03-L07

In this exercise, you will apply your knowledge of Deep Reinforcement Learning, specifically the Proximal Policy Optimization (PPO) algorithm, to train an agent to solve a continuous control task. We will use the `BipedalWalker-v3` environment from the Gymnasium library, a classic benchmark for continuous control where the agent must learn to walk without falling.

### Task Description

Your goal is to implement and train a PPO agent that can successfully navigate the `BipedalWalker-v3` environment. The agent should learn to walk efficiently and achieve a high average reward over multiple evaluation episodes.

### Requirements

1.  **Environment Setup**: Initialize the `BipedalWalker-v3` environment. For efficient training, use a vectorized environment (e.g., `VecMonitor` and `DummyVecEnv` from Stable Baselines3).
2.  **PPO Agent Implementation**: Utilize the `PPO` implementation from the `stable_baselines3` library. You will need to define a suitable Multi-Layer Perceptron (MLP) policy network architecture.
3.  **Training**: Train the PPO agent for a sufficient number of timesteps (e.g., 2-5 million timesteps) to observe significant learning.
4.  **Logging and Monitoring**: Integrate TensorBoard logging to track training progress, including episode rewards, policy loss, value loss, and entropy.
5.  **Evaluation**: After training, evaluate the agent's performance over a set number of episodes (e.g., 10-20 episodes) and report the average reward and standard deviation.
6.  **Visualization**: Plot the training reward curve (e.g., using data from TensorBoard logs or a custom callback).
7.  **Code Quality**: Ensure your code is clean, well-commented, and follows Python best practices.

### Evaluation Criteria

*   **Successful Training**: The agent demonstrates clear learning and achieves an average reward above a baseline (e.g., > 100 for `BipedalWalker-v3` is a good start, ideally aiming for > 200-300).
*   **Correct PPO Usage**: Proper instantiation and configuration of the `stable_baselines3.PPO` model.
*   **Effective Logging**: TensorBoard logs are correctly set up and provide meaningful insights into training.
*   **Clear Evaluation**: The evaluation process is robust, and results are clearly presented.
*   **Readability**: Code is easy to understand and well-documented.

Good luck!


In [ ]:
# Core Libraries
import gymnasium as gym
import numpy as np
import torch
import matplotlib.pyplot as plt
import os

# Stable Baselines3 for PPO and utilities
from stable_baselines3 import PPO
from stable_baselines3.common.env_util import make_vec_env
from stable_baselines3.common.vec_env import VecMonitor, DummyVecEnv
from stable_baselines3.common.callbacks import EvalCallback, StopTrainingOnRewardThreshold
from stable_baselines3.common.results_plotter import load_results, ts2xy

# Ensure reproducibility
SEED = 42
np.random.seed(SEED)
torch.manual_seed(SEED)

# --- Environment Setup ---
ENV_ID = "BipedalWalker-v3"
LOG_DIR = "./ppo_bipedal_walker_logs/"
EVAL_FREQ = 10000 # Evaluate every N timesteps
N_EVAL_EPISODES = 10 # Number of episodes for evaluation

# Create log directory if it doesn't exist
os.makedirs(LOG_DIR, exist_ok=True)

print(f"Environment: {ENV_ID}")
print(f"Logging to: {LOG_DIR}")

# Helper function to create a vectorized environment
def make_env():
    env = gym.make(ENV_ID)
    return env

# Create a vectorized environment for training
# DummyVecEnv is used for single-process vectorized environments, suitable for simpler setups.
# For more complex scenarios or larger number of environments, SubprocVecEnv is often preferred.
vec_env = make_vec_env(make_env, n_envs=4, seed=SEED)
vec_env = VecMonitor(vec_env, LOG_DIR)

# --- PPO Hyperparameters (initial suggestions) ---
# These are common starting points for BipedalWalker-v3, but tuning is often required.
# For a more robust solution, one might use Optuna or other hyperparameter optimization tools.
ppo_params = {
    "learning_rate": 3e-4,
    "n_steps": 2048, # Number of steps to run for each environment per update
    "batch_size": 64, # Minibatch size for SGD
    "n_epochs": 10, # Number of epochs for the policy and value networks
    "gamma": 0.99, # Discount factor
    "gae_lambda": 0.95, # Factor for trade-off of bias vs variance for Generalized Advantage Estimator
    "clip_range": 0.2, # Clipping parameter for PPO
    "ent_coef": 0.01, # Entropy coefficient for exploration
    "vf_coef": 0.5, # Value function coefficient for the loss calculation
    "max_grad_norm": 0.5, # The maximum value for the gradient clipping
    "policy_kwargs": dict(activation_fn=torch.nn.Tanh, net_arch=dict(pi=[256, 256], vf=[256, 256]))
}

print("Initial PPO Hyperparameters:")
for k, v in ppo_params.items():
    print(f"  {k}: {v}")

print("Setup complete. Ready for agent implementation.")


## Your Turn: Implement and Train the PPO Agent

Now it's your turn to put everything together. Using the provided setup code, implement the following:

1.  **Instantiate the PPO model**: Create an instance of `stable_baselines3.PPO` using the `vec_env` and the `ppo_params` defined above. Remember to set `verbose=1` to see training progress.
2.  **Define Callbacks**: Implement an `EvalCallback` to periodically evaluate the agent's performance on a separate evaluation environment. This callback should also save the best model found during training. Optionally, add a `StopTrainingOnRewardThreshold` to halt training once a satisfactory performance is reached.
3.  **Train the Agent**: Call the `learn()` method on your PPO model for a total of `total_timesteps` (e.g., 2-5 million).
4.  **Load and Evaluate the Best Model**: After training, load the best model saved by the `EvalCallback` and evaluate its performance over `N_EVAL_EPISODES`.
5.  **Visualize Training**: Plot the mean reward over time using the data collected by `VecMonitor` (which saves to `LOG_DIR`).

Feel free to experiment with hyperparameters or network architectures if you wish to achieve better performance, but start with the provided suggestions.


In [ ]:
# --- Reference Solution --- 

# 1. Instantiate the PPO model
# We'll use the MlpPolicy, which is the default for continuous action spaces.
# The policy_kwargs define the neural network architecture for both actor (pi) and critic (vf).
model = PPO(
    "MlpPolicy",
    vec_env,
    verbose=1,
    tensorboard_log=LOG_DIR,
    seed=SEED,
    **ppo_params
)

print("PPO Model instantiated successfully.")

# 2. Define Callbacks
# Create a separate evaluation environment
eval_env = make_vec_env(make_env, n_envs=1, seed=SEED + 1) # Use a different seed for eval
eval_env = VecMonitor(eval_env, LOG_DIR + "/eval_results/")

# Callback to stop training when a reward threshold is met
# BipedalWalker-v3 is considered solved at 300+ reward.
stop_callback = StopTrainingOnRewardThreshold(reward_threshold=250.0, verbose=1)

# Callback for evaluation and saving the best model
# It will save the model that achieves the highest mean reward on the evaluation environment.
eval_callback = EvalCallback(
    eval_env,
    best_model_save_path=LOG_DIR + "/best_model/",
    log_path=LOG_DIR + "/eval_results/",
    eval_freq=EVAL_FREQ,
    n_eval_episodes=N_EVAL_EPISODES,
    deterministic=True, # Use deterministic actions during evaluation
    render=False, # Do not render during evaluation
    callback_on_new_best=stop_callback, # Stop training if new best exceeds threshold
    verbose=1
)

# Combine callbacks if needed, though EvalCallback can handle StopTrainingOnRewardThreshold internally
# from stable_baselines3.common.callbacks import CallbackList
# callback_list = CallbackList([eval_callback])

print("Callbacks defined.")

# 3. Train the Agent
TOTAL_TIMESTEPS = 3_000_000 # Total timesteps for training
print(f"Starting training for {TOTAL_TIMESTEPS} timesteps...")

try:
    model.learn(
        total_timesteps=TOTAL_TIMESTEPS,
        callback=eval_callback,
        progress_bar=True # Show a nice progress bar
    )
    print("Training finished.")
except KeyboardInterrupt:
    print("Training interrupted by user.")

# Save the final model
model.save(LOG_DIR + "/final_model")
print("Final model saved.")

# 4. Load and Evaluate the Best Model
print("\n--- Evaluating Best Model ---")

# Load the best model found during training
# If no best model was saved (e.g., training stopped early), load the final model.
if os.path.exists(LOG_DIR + "/best_model/best_model.zip"):
    best_model = PPO.load(LOG_DIR + "/best_model/best_model", env=vec_env)
    print("Loaded best model from training.")
else:
    best_model = model # Use the last trained model if no 'best' was saved
    print("No best model found, using the final trained model.")

# Evaluate the loaded model
episode_rewards = []
for i in range(N_EVAL_EPISODES):
    obs, info = eval_env.reset()
    done = False
    episode_reward = 0
    while not done:
        action, _states = best_model.predict(obs, deterministic=True)
        obs, reward, terminated, truncated, info = eval_env.step(action)
        done = terminated or truncated
        episode_reward += reward[0] # Reward is an array for vectorized envs
    episode_rewards.append(episode_reward)

mean_reward = np.mean(episode_rewards)
std_reward = np.std(episode_rewards)

print(f"\nEvaluation Results (over {N_EVAL_EPISODES} episodes):")
print(f"  Mean Reward: {mean_reward:.2f}")
print(f"  Standard Deviation of Reward: {std_reward:.2f}")

# 5. Visualize Training
print("\n--- Plotting Training Rewards ---")

# Load results from the monitor wrapper
x, y = ts2xy(load_results(LOG_DIR), 'timesteps')

if len(x) > 0:
    plt.figure(figsize=(12, 6))
    plt.plot(x, y)
    plt.xlabel('Timesteps')
    plt.ylabel('Mean Episode Reward')
    plt.title(f'PPO Training on {ENV_ID}')
    plt.grid(True)
    plt.show()
else:
    print("No data to plot. Ensure VecMonitor logs are being saved correctly.")

# Clean up environment
vec_env.close()
eval_env.close()
print("Environment closed.")
